# City of Cape Town Service Request Data Quality

This notebook analyzes City of Cape Town service request data for data quality and formatting issues that may affect downstream analysis.

**Data Sources:**
- Service request data (`sr_hex.csv`)
- H3 hex polygons (`hex_polygons.geojson`)

**Contents:**
1. [Setup & Configuration](#setup)
2. [Load Data](#load-data)
3. [Data Quality Report](#quality-report)
4. [Validation Checks](#validation)
5. [Sample Invalid Records](#samples)
6. [H3 to Suburb Mapping Analysis](#h3-mapping)

<a id="setup"></a>
## 1. Setup & Configuration

In [18]:
import warnings
from datetime import datetime
from io import StringIO

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

<a id="load-data"></a>
## 2. Load Data

**Expected schema:**
| Field | Type | Constraints |
|-------|------|-------------|
| `notification_number` | string | 12 digits, not nullable |
| `reference_number` | string | 10 digits, nullable |
| `creation_timestamp` | timestamp | +02:00 timezone |
| `completion_timestamp` | timestamp | +02:00 timezone, nullable |
| `latitude` | float | between -35 and -32 |
| `longitude` | float | between 17 and 20 |
| `h3_level8_index` | string | 15 hex chars OR "0", not nullable |

In [19]:
# File paths
DATA_PATH = 'raw_data/sr_hex.csv'
GEO_PATH = 'raw_data/hex_polygons.geojson'

# Column definitions
EXPECTED_COLUMNS = [
    'notification_number', 'reference_number',
    'creation_timestamp', 'completion_timestamp',
    'directorate', 'department', 'branch', 'section',
    'code_group', 'code', 'cause_code_group', 'cause_code',
    'official_suburb', 'latitude', 'longitude', 'h3_level8_index',
]

DTYPE_SPEC = {
    'notification_number': str, 'reference_number': str,
    'directorate': str, 'department': str, 'branch': str, 'section': str,
    'code_group': str, 'code': str, 'cause_code_group': str, 'cause_code': str,
    'official_suburb': str, 'h3_level8_index': str,
}

# Capture malformed CSV lines during parsing
malformed_lines = []

def capture_bad_lines(bad_line):
    malformed_lines.append(bad_line)
    return None

df_raw = pd.read_csv(
    DATA_PATH,
    dtype=DTYPE_SPEC,
    names=EXPECTED_COLUMNS,
    header=0,
    on_bad_lines=capture_bad_lines,
    engine='python'
)

print(f"Loaded {len(df_raw):,} records")
print(f"Malformed lines: {len(malformed_lines)}")

Loaded 941,633 records
Malformed lines: 1


In [20]:
if malformed_lines:
    print(f"Total malformed lines: {len(malformed_lines)}\n")
    for i, line in enumerate(malformed_lines[:5]):
        print(f"--- Line {i+1} ---\n{line}")
else:
    print("No malformed lines detected.")

Total malformed lines: 1

--- Line 1 ---
['000400538915', 'FROGGY!', 'FROGGY!', 'FROGGY!', '9108601346', '2020-03-19 06:36:06+02:00', '2021-03-29 20:34:19+02:00', 'URBAN MOBILITY', 'Roads Infrastructure Management', 'RIM Area North', 'District : Bellville', 'TD Customer complaint groups', 'Paint Markings Lines&Signs', 'Road Markings', 'Wear and tear', 'RAVENSMEAD', '-33.9200192857365', '18.607209458428894', '88ad361133fffff']


<a id="quality-report"></a>
## 3. Data Quality Report

Overview of null rates and value ranges across all fields.

In [21]:
def generate_quality_report(df):
    """Generate a summary of null rates and numeric ranges."""
    lines = [
        f"Total Records: {len(df):,}",
        f"Total Columns: {len(df.columns)}",
        "",
        "NULL RATES PER FIELD",
        "-" * 55,
    ]
    
    null_rates = df.isnull().sum() / len(df) * 100
    for col in df.columns:
        null_count = df[col].isnull().sum()
        lines.append(f"{col:30} | {null_count:>8,} nulls | {null_rates[col]:>6.2f}%")
    
    lines.extend(["", "NUMERIC FIELD RANGES", "-" * 55])
    for col in ['latitude', 'longitude']:
        if col in df.columns:
            lines.append(f"{col:30} | min: {df[col].min():>12.6f} | max: {df[col].max():>12.6f}")
    
    return "\n".join(lines)

print(generate_quality_report(df_raw))

Total Records: 941,633
Total Columns: 16

NULL RATES PER FIELD
-------------------------------------------------------
notification_number            |        0 nulls |   0.00%
reference_number               |  348,714 nulls |  37.03%
creation_timestamp             |        0 nulls |   0.00%
completion_timestamp           |   12,192 nulls |   1.29%
directorate                    |    9,435 nulls |   1.00%
department                     |    9,454 nulls |   1.00%
branch                         |   28,401 nulls |   3.02%
section                        |   93,125 nulls |   9.89%
code_group                     |        0 nulls |   0.00%
code                           |        0 nulls |   0.00%
cause_code_group               |  810,517 nulls |  86.08%
cause_code                     |  811,965 nulls |  86.23%
official_suburb                |  212,413 nulls |  22.56%
latitude                       |  212,364 nulls |  22.55%
longitude                      |  212,364 nulls |  22.55%
h3_level8_i

<a id="validation"></a>
## 4. Validation Checks

Validate records against schema constraints and business rules.

In [22]:
def validate_data(df):
    """
    Run validation checks against schema constraints.
    
    Note: Parses creation_timestamp to datetime (modifies df in place).
    Returns dict of issue counts and the modified dataframe.
    """
    issues = {}
    
    # Notification number: must be 12 digits
    invalid_notif = df['notification_number'].isnull() | ~df['notification_number'].str.match(r'^\d{12}$', na=False)
    issues['invalid_notification_number'] = invalid_notif.sum()
    
    # Parse timestamps
    original_creation = df['creation_timestamp'].copy()
    df['creation_timestamp'] = pd.to_datetime(df['creation_timestamp'], errors='coerce', utc=True)
    
    unparseable = df['creation_timestamp'].isna() & original_creation.notna()
    issues['unparseable_creation_timestamp'] = unparseable.sum()
    
    # Timestamp range check (2020-01-01 to now)
    cutoff_date = pd.Timestamp('2020-01-01', tz='Africa/Johannesburg')
    future_date = pd.Timestamp.now(tz='Africa/Johannesburg')
    out_of_range = df['creation_timestamp'].notna() & (
        (df['creation_timestamp'] < cutoff_date) | (df['creation_timestamp'] > future_date)
    )
    issues['creation_timestamp_out_of_range'] = out_of_range.sum()
    
    # Latitude bounds: -35 to -32
    issues['out_of_bounds_latitude'] = ((df['latitude'] < -35) | (df['latitude'] > -32)).sum()
    issues['null_latitude'] = df['latitude'].isnull().sum()
    
    # Longitude bounds: 17 to 20
    issues['out_of_bounds_longitude'] = ((df['longitude'] < 17) | (df['longitude'] > 20)).sum()
    issues['null_longitude'] = df['longitude'].isnull().sum()
    
    # H3 index: 15 hex chars or "0"
    invalid_h3 = df['h3_level8_index'].isnull() | ~df['h3_level8_index'].str.match(r'^([0-9a-fA-F]{15}|0)$', na=False)
    issues['invalid_h3_index'] = invalid_h3.sum()
    
    return issues, df

issues, df = validate_data(df_raw.copy())

print("VALIDATION ISSUES FOUND")
print("-" * 55)
for issue_type, count in issues.items():
    pct = count / len(df_raw) * 100
    print(f"{issue_type:40} | {count:>8,} | {pct:>5.2f}%")

VALIDATION ISSUES FOUND
-------------------------------------------------------
invalid_notification_number              |        0 |  0.00%
unparseable_creation_timestamp           |        0 |  0.00%
creation_timestamp_out_of_range          |        0 |  0.00%
out_of_bounds_latitude                   |        0 |  0.00%
null_latitude                            |  212,364 | 22.55%
out_of_bounds_longitude                  |        0 |  0.00%
null_longitude                           |  212,364 | 22.55%
invalid_h3_index                         |        0 |  0.00%


<a id="samples"></a>
## 5. Sample Invalid Records

Display sample records for each type of validation failure.

In [23]:
SAMPLE_COLS = [
    'notification_number', 'creation_timestamp', 'latitude', 'longitude',
    'code_group', 'code', 'official_suburb', 'h3_level8_index'
]

def find_invalid_records(df_raw, df_parsed):
    """Identify records with validation issues by category."""
    samples = {}
    
    # Out-of-bounds coordinates
    oob = df_raw[
        ((df_raw['latitude'] < -35) | (df_raw['latitude'] > -32)) |
        ((df_raw['longitude'] < 17) | (df_raw['longitude'] > 20))
    ]
    if len(oob) > 0:
        samples['Out-of-bounds coordinates'] = oob
    
    # Timestamps out of range
    cutoff = pd.Timestamp('2020-01-01', tz='Africa/Johannesburg')
    future = pd.Timestamp.now(tz='Africa/Johannesburg')
    bad_dates = df_parsed[
        df_parsed['creation_timestamp'].notna() &
        ((df_parsed['creation_timestamp'] < cutoff) | (df_parsed['creation_timestamp'] > future))
    ]
    if len(bad_dates) > 0:
        samples['Timestamps out of range'] = bad_dates
    
    # Null coordinates
    null_coords = df_raw[df_raw['latitude'].isnull() | df_raw['longitude'].isnull()]
    if len(null_coords) > 0:
        samples['Null coordinates'] = null_coords
    
    # Invalid H3 index
    invalid_h3 = df_raw[
        df_raw['h3_level8_index'].isnull() |
        ~df_raw['h3_level8_index'].str.match(r'^([0-9a-fA-F]{15}|0)$', na=False)
    ]
    if len(invalid_h3) > 0:
        samples['Invalid H3 index'] = invalid_h3
    
    return samples

samples = find_invalid_records(df_raw, df)

if samples:
    for issue, df_issue in samples.items():
        print(f"\n{issue}: {len(df_issue):,} record(s)")
        display(df_issue[SAMPLE_COLS].head())
else:
    print("No records with validation issues.")


Null coordinates: 212,364 record(s)


,notification_number,creation_timestamp,latitude,longitude,code_group,code,official_suburb,h3_level8_index
13741,000400525315,2020-01-23 13:28:51+02:00,NaN,NaN,TD Customer complaint groups,"RequestNewRoadway painted, mounted signs",NaN,0
13742,000400527116,2020-01-30 12:46:49+02:00,NaN,NaN,TD Customer complaint groups,"RequestNewRoadway painted, mounted signs",NaN,0
13743,000400528840,2020-02-06 12:29:29+02:00,NaN,NaN,TD Customer complaint groups,"RequestNewRoadway painted, mounted signs",NaN,0
13744,000400530412,2020-02-12 08:38:03+02:00,NaN,NaN,TD Customer complaint groups,"RequestNewRoadway painted, mounted signs",NaN,0
13745,000400530772,2020-02-13 09:27:42+02:00,NaN,NaN,TD Customer complaint groups,Paint Markings Lines&Signs,NaN,0


<a id="h3-mapping"></a>
## 6. H3 to Suburb Mapping Analysis

Check whether the relationship between `official_suburb` and `h3_level8_index` is one-to-one or many-to-many.

In [25]:
mapping = df[['official_suburb', 'h3_level8_index']].dropna().drop_duplicates()

suburb_h3_counts = mapping.groupby('official_suburb')['h3_level8_index'].nunique()
h3_suburb_counts = mapping.groupby('h3_level8_index')['official_suburb'].nunique()

multi_hex_suburbs = (suburb_h3_counts > 1).sum()
multi_suburb_hexes = (h3_suburb_counts > 1).sum()

print("SUBURB <-> H3 INDEX MAPPING")
print("-" * 55)
print(f"Suburbs mapped to multiple hexes: {multi_hex_suburbs}")
print(f"Hexes mapped to multiple suburbs: {multi_suburb_hexes}")
print(f"\nConclusion: {'Many-to-many relationship' if multi_hex_suburbs and multi_suburb_hexes else '1:1 relationship'}")

print("\nSample suburbs with >1 H3 index:")
print(suburb_h3_counts[suburb_h3_counts > 1].head().to_string())

print("\nSample H3 indices with >1 suburb:")
print(h3_suburb_counts[h3_suburb_counts > 1].head().to_string())

# Additional test: find h3_level8_index values with no corresponding official_suburb value
h3_no_suburb = df[df['official_suburb'].isnull() & df['h3_level8_index'].notnull()]
num_h3_no_suburb = h3_no_suburb['h3_level8_index'].nunique()
print(f"\nH3 indices with no corresponding suburb: {num_h3_no_suburb}")
if num_h3_no_suburb > 0:
    print("Sample H3 indices without suburb mapping:")
    print(h3_no_suburb['h3_level8_index'].drop_duplicates().head().to_string(index=False))

SUBURB <-> H3 INDEX MAPPING
-------------------------------------------------------
Suburbs mapped to multiple hexes: 714
Hexes mapped to multiple suburbs: 1127

Conclusion: Many-to-many relationship

Sample suburbs with >1 H3 index:
official_suburb
AAN DE WIJNLANDEN ESTATE     3
ACACIA PARK                 10
ADMIRALS PARK                2
ADRIAANSE                    4
AIRPORT                      2

Sample H3 indices with >1 suburb:
h3_level8_index
88ad360001fffff    2
88ad360005fffff    2
88ad360007fffff    2
88ad36000dfffff    2
88ad360021fffff    4

H3 indices with no corresponding suburb: 23
Sample H3 indices without suburb mapping:
88ad36196bfffff
88ad36104dfffff
              0
88ad368899fffff
88ad361529fffff


I don't love that there are `h3_level8_index` values with no associated `official_suburb` value. 
I had been treating `official_suburb` as a highly reliable field.
But at 49 records (see below) I'm going to ignore it as immaterial.

In [28]:
sample = df_raw[(df_raw['h3_level8_index'].astype(str) != '0') & (df_raw['official_suburb'].isnull())]
count = len(sample)
print(f"Total number of records where h3_level8_index is NOT 0 and official_suburb is missing: {count:,}")
print("Sample records (showing up to 5 rows):")
display(sample.head(5))

Total number of records where h3_level8_index is NOT 0 and official_suburb is missing: 49
Sample records (showing up to 5 rows):


,notification_number,reference_number,creation_timestamp,completion_timestamp,directorate,department,branch,section,code_group,code,cause_code_group,cause_code,official_suburb,latitude,longitude,h3_level8_index
102,000400577592,9109381408,2020-09-17 11:11:45+02:00,2021-06-16 13:35:14+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area North,District : Kraaifontein,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,NaN,NaN,NaN,-33.834225,18.752790,88ad36196bfffff
942,000400569291,NaN,2020-08-26 14:28:29+02:00,2020-10-01 15:10:58+02:00,URBAN MOBILITY,Roads Infrastructure Management,RIM Area South,District : Athlone,TD Customer complaint groups,Pothole&Defect Road Foot Bic Way/Kerbs,Road (RCL),Surfacing failure,NaN,-33.994574,18.591700,88ad36104dfffff
45403,001015475420,9108207768,2020-01-06 11:12:23+02:00,2020-01-06 11:25:14+02:00,ENERGY,Electricity Generation and Distribution,Electricity Retail Management,Customer Support Services and Rev Man,ELECTRICITY TECHNICAL COMPLAINTS,Sparks on Pole,NaN,NaN,NaN,-34.088274,18.554771,88ad368899fffff
54160,001015485314,9108224779,2020-01-08 16:22:51+02:00,2020-01-10 08:17:09+02:00,WATER AND SANITATION,Distribution Services,Reticulation,Reticulation WW Conveyance,SEWER,Sewer: Blocked/Overflow,NaN,NaN,NaN,-33.912280,18.463431,88ad361529fffff
89260,001015525048,9108284595,2020-01-21 07:36:54+02:00,2020-01-21 09:13:36+02:00,ENERGY,Electricity Generation and Distribution,Electricity Retail Management,Customer Support Services and Rev Man,ELECTRICITY FINANCIAL AND METER READING,Incorrect Reading/Request Actual Reading,NaN,NaN,NaN,-34.205123,18.458149,88ad3680d7fffff
